Step 1- Import Libraries

In [1]:
import pandas as pd
import numpy as np

Step 2-Load Cleaned datset 


In [2]:

df = pd.read_csv("../data/Processed/Upi_transactions_cleaned.csv")

In [3]:
print("Shape:",df.shape)

Shape: (10000, 18)


Step 3- Convert Timestamp

In [4]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"],errors ="coerce")


In [5]:
print(df["Timestamp"].dtype)


datetime64[ns]


Step 4 -Sort Transactions

In [6]:
df =df.sort_values(["Sender_ID","Timestamp"]).reset_index(drop=True)

In [7]:
df

,Transaction_ID,Timestamp,Sender_ID,Receiver_ID,Amount,Merchant_Category,Transaction_Type,Location,Device_Type,UPI_Channel,Hour_of_Day,Day_of_Week,Historical_Spend_Pattern,Spend_Deviation_Ratio,Risk_Label,Anomaly_Flag,Anomaly_Reason,Is_Weekend
0,TXN102315,2026-02-12 06:05:18,U1000,R1034,805.48,Grocery,P2M,Noida,iOS,UPI Lite,6,Thursday,1088.03,0.740,Low,0,Normal,0
1,TXN103673,2026-03-09 03:16:53,U1000,R1044,325.77,Fuel,Online Purchase,Surat,Android,UPI App,3,Monday,1088.03,0.299,Low,0,Normal,0
2,TXN105082,2026-04-03 18:16:39,U1000,R1520,1541.69,E-commerce,P2M,Surat,Android,UPI App,18,Friday,1088.03,1.417,Low,0,Normal,0
3,TXN106301,2026-04-26 01:34:52,U1000,R1410,948.36,Fuel,P2P,Noida,iOS,UPI App,1,Sunday,1088.03,0.872,Low,0,Normal,1
4,TXN107005,2026-05-08 14:35:53,U1000,R1129,1492.54,Shopping,Online Purchase,Bengaluru,Android,USSD,14,Friday,1088.03,1.372,Low,0,Normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,TXN108146,2026-05-28 22:13:30,U2199,R1103,668.89,Utilities,P2M,Bengaluru,Android,QR Code,22,Thursday,422.10,1.585,Low,0,Normal,0
9996,TXN108702,2026-06-08 00:28:34,U2199,R1342,195.83,Food & Dining,Online Purchase,Delhi,iOS,UPI App,0,Monday,422.10,0.464,Low,0,Normal,0
9997,TXN109592,2026-06-23 21:34:23,U2199,R1643,1165.29,Fuel,P2M,Surat,iOS,UPI App,21,Tuesday,422.10,2.761,Low,0,Normal,0
9998,TXN109636,2026-06-24 14:37:39,U2199,R1476,1111.46,Financial Services,Online Purchase,Hyderabad,Android,UPI App,14,Wednesday,422.10,2.633,Medium,0,Normal,0


Step 5 - Time_Based Features

5.1 Hour

In [8]:
df["Hour"] = df["Timestamp"].dt.hour

5.2 -Day of Week

In [9]:
df["Day of Week"] =df["Timestamp"].dt.dayofweek

5.3 Night Transaction

In [10]:
df["Is_Night"] = (
    (df["Hour"] >= 0) &
    (df["Hour"] < 6)
).astype(int)

Step 6 — User Average Transaction Amount

In [11]:
df["User_Avg_Amount"] = (df.groupby("Sender_ID")["Amount"].transform("mean"))

In [12]:
df[[ "Sender_ID","Amount","User_Avg_Amount" ]].head(10)

,Sender_ID,Amount,User_Avg_Amount
0,U1000,805.48,3120.610000
1,U1000,325.77,3120.610000
2,U1000,1541.69,3120.610000
3,U1000,948.36,3120.610000
4,U1000,1492.54,3120.610000
5,U1000,13609.82,3120.610000
6,U1001,497.49,715.416667
7,U1001,290.51,715.416667
8,U1001,910.72,715.416667
9,U1001,944.65,715.416667


Step 7 — Amount Deviation

How different is the current transaction from the user's average transaction amount?

In [13]:
df["Amount_Deviation"] = (df["Amount"] / df["User_Avg_Amount"])

Step 8 — Difference From User Average

In [14]:
df["Amount_Difference"] = (df["Amount"] - df["User_Avg_Amount"])

Step 9 — Transaction Frequency

In [15]:
df["Transactions_Per_Hour"] = (
    df.groupby(
        ["Sender_ID",df["Timestamp"].dt.floor("h")]
    )["Transaction_ID"].transform("count"))

Step 10 — Time Since Previous Transaction

In [16]:
df["Previous_Timestamp"] = (
    df.groupby("Sender_ID")["Timestamp"]
    .shift(1)
)

In [17]:
df["Time_Since_Last_Transaction"] = (
    df["Timestamp"] -
    df["Previous_Timestamp"]
).dt.total_seconds() / 60

Step 11 — Recipient Frequency

In [18]:
df["Recipiet_Frequency"] = (
    df.groupby(
        ["Sender_ID","Receiver_ID"]
    )["Transaction_ID"].transform("count")
)

Step 12 — Device Change

In [20]:
df["Previous_Device"] = (
    df.groupby("Sender_ID")
    ["Device_Type"]
    .shift(1)
)
    

In [24]:
df["Device_Change"] = (
    (df["Device_Type"] != df["Previous_Device"])
    &
    df["Previous_Device"].notna()
).astype(int)


Step 13 Location Change

In [27]:
df["Previous_Location"] = (
    df.groupby("Sender_ID")
    ["Location"].shift(1)
)

In [29]:
df["Location_Change"] = (
    (
        df["Location"] != df["Previous_Location"]
    ) &
    df["Previous_Location"].notna()
).astype(int)

Step 14 — Remove Temporary Columns

In [30]:
df = df.drop(
    columns=[
        "Previous_Timestamp",
        "Previous_Device",
        "Previous_Location"
    ]
)

Step 15 — Feature Review

In [31]:
df.head()

,Transaction_ID,Timestamp,Sender_ID,Receiver_ID,Amount,Merchant_Category,Transaction_Type,Location,Device_Type,UPI_Channel,...,Day of Week,Is_Night,User_Avg_Amount,Amount_Deviation,Amount_Difference,Transactions_Per_Hour,Time_Since_Last_Transaction,Recipiet_Frequency,Device_Change,Location_Change
0,TXN102315,2026-02-12 06:05:18,U1000,R1034,805.48,Grocery,P2M,Noida,iOS,UPI Lite,...,3,0,3120.61,0.258116,-2315.13,1,NaN,1,0,0
1,TXN103673,2026-03-09 03:16:53,U1000,R1044,325.77,Fuel,Online Purchase,Surat,Android,UPI App,...,0,1,3120.61,0.104393,-2794.84,1,35831.583333,1,1,1
2,TXN105082,2026-04-03 18:16:39,U1000,R1520,1541.69,E-commerce,P2M,Surat,Android,UPI App,...,4,0,3120.61,0.494035,-1578.92,1,36899.766667,1,0,0
3,TXN106301,2026-04-26 01:34:52,U1000,R1410,948.36,Fuel,P2P,Noida,iOS,UPI App,...,6,1,3120.61,0.303902,-2172.25,1,32118.216667,1,1,1
4,TXN107005,2026-05-08 14:35:53,U1000,R1129,1492.54,Shopping,Online Purchase,Bengaluru,Android,USSD,...,4,0,3120.61,0.478285,-1628.07,1,18061.016667,1,1,1


In [35]:
df = df.rename(columns={
    "Day of Week": "Day_of_Week",
    "Recipiet_Frequency": "Recipient_Frequency"
})

In [36]:
feature_columns = [
    "Hour",
    "Day_of_Week",
    "Is_Weekend",
    "Is_Night",
    "User_Avg_Amount",
    "Amount_Deviation",
    "Amount_Difference",
    "Transactions_Per_Hour",
    "Time_Since_Last_Transaction",
    "Recipient_Frequency",
    "Device_Change",
    "Location_Change"
]

df[feature_columns].head(10)

,Hour,Day_of_Week,Day_of_Week,Is_Weekend,Is_Night,User_Avg_Amount,Amount_Deviation,Amount_Difference,Transactions_Per_Hour,Time_Since_Last_Transaction,Recipient_Frequency,Device_Change,Location_Change
0,6,Thursday,3,0,0,3120.610000,0.258116,-2315.130000,1,NaN,1,0,0
1,3,Monday,0,0,1,3120.610000,0.104393,-2794.840000,1,35831.583333,1,1,1
2,18,Friday,4,0,0,3120.610000,0.494035,-1578.920000,1,36899.766667,1,0,0
3,1,Sunday,6,1,1,3120.610000,0.303902,-2172.250000,1,32118.216667,1,1,1
4,14,Friday,4,0,0,3120.610000,0.478285,-1628.070000,1,18061.016667,1,1,1
5,11,Sunday,6,1,0,3120.610000,4.361269,10489.210000,1,32956.050000,1,0,1
6,10,Tuesday,1,0,0,715.416667,0.695385,-217.926667,1,NaN,1,0,0
7,17,Wednesday,2,0,0,715.416667,0.406071,-424.906667,1,1815.550000,1,0,1
8,18,Wednesday,2,0,0,715.416667,1.272992,195.303333,1,95.933333,1,1,1
9,2,Friday,4,0,1,715.416667,1.320419,229.233333,1,11980.616667,1,0,0


Step 16 — Feature Information

In [37]:
df[feature_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Hour                         10000 non-null  int32  
 1   Day_of_Week                  10000 non-null  object 
 2   Day_of_Week                  10000 non-null  int32  
 3   Is_Weekend                   10000 non-null  int64  
 4   Is_Night                     10000 non-null  int32  
 5   User_Avg_Amount              10000 non-null  float64
 6   Amount_Deviation             10000 non-null  float64
 7   Amount_Difference            10000 non-null  float64
 8   Transactions_Per_Hour        10000 non-null  int64  
 9   Time_Since_Last_Transaction  8800 non-null   float64
 10  Recipient_Frequency          10000 non-null  int64  
 11  Device_Change                10000 non-null  int32  
 12  Location_Change              10000 non-null  int32  
dtypes: float64(4), in

In [38]:
df[feature_columns].describe()

,Hour,Day_of_Week,Is_Weekend,Is_Night,User_Avg_Amount,Amount_Deviation,Amount_Difference,Transactions_Per_Hour,Time_Since_Last_Transaction,Recipient_Frequency,Device_Change,Location_Change
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,10000.000000,8800.000000,10000.000000,10000.000000,10000.000000
mean,11.44130,2.982300,0.280400,0.255000,1549.603755,1.000000,3.819878e-15,1.001400,26903.993398,1.008600,0.370700,0.815900
std,6.95002,2.001146,0.449217,0.435883,1438.601259,0.787722,2.311625e+03,0.037392,26356.806436,0.092341,0.483016,0.387585
min,0.00000,0.000000,0.000000,0.000000,115.736000,0.025206,-3.333908e+04,1.000000,8.650000,1.000000,0.000000,0.000000
25%,5.00000,1.000000,0.000000,0.000000,736.878750,0.502788,-5.894508e+02,1.000000,7880.270833,1.000000,0.000000,1.000000
50%,12.00000,3.000000,0.000000,0.000000,1159.690909,0.810362,-1.651311e+02,1.000000,18925.183333,1.000000,0.000000,1.000000
75%,17.00000,5.000000,1.000000,1.000000,1928.518750,1.252078,2.295167e+02,1.000000,37320.600000,1.000000,1.000000,1.000000
max,23.00000,6.000000,1.000000,1.000000,37911.040000,9.812291,6.610519e+04,2.000000,212467.233333,2.000000,1.000000,1.000000


df[feature_columns].describe()

In [39]:
output_path = "../data/processed/upi_transactions_feature_engineered.csv"
df.to_csv(output_path,index =False)

print("Shape:", df.shape)

Shape: (10000, 29)
